# Notebook 06 — Side-by-side model comparison

Runs the **same questions** through multiple models and prints the answers
next to each other. This is the core experiment for your report.

**Models compared:**
- `own_no_rag`  — your trained transformer, no retrieved context
- `own_rag`     — your trained transformer + RAG
- `flan_rag`    — flan-t5-base + RAG (strong pretrained baseline)
- `gpt4`        — GPT-4o-mini via API (frontier baseline)

Load only the ones you want to test — comment out the rest in the config cell.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CONFIGURATION
# Comment out any models you don't want to run.
# ═══════════════════════════════════════════════════════════

MODELS_TO_RUN = [
    "own_no_rag",   # your model, no RAG
    "own_rag",      # your model + RAG
    "flan_rag",     # flan-t5-base + RAG
    # "gpt4",       # GPT-4o-mini (requires OPENAI_API_KEY)
]

CHECKPOINT_PATH = "../teaching_assistant/checkpoints/step_005000.pt"
RAG_INDEX_DIR   = "../teaching_assistant/rag_index"
OPENAI_API_KEY  = "sk-..."   # only needed if "gpt4" is in MODELS_TO_RUN

# Questions to compare across models
QUESTIONS = [
    "What is dropout and why does it help with overfitting?",
    "Explain backpropagation in simple terms.",
    "What is the difference between batch normalization and layer normalization?",
]

MAX_TOKENS  = 200
TEMPERATURE = 0.7

In [ ]:
# ═══════════════════════════════════════════════════════════
# LOAD ALL REQUESTED MODELS
# Each model is stored as a callable: models[name](question) → str
# ═══════════════════════════════════════════════════════════
import torch, sys
sys.path.insert(0, "../teaching_assistant")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

models   = {}   # name → callable(question) → str
retriever = None

# ── Load RAG retriever (shared by all RAG models) ──────────
if any("rag" in m for m in MODELS_TO_RUN):
    try:
        from retriever import Retriever
        retriever = Retriever(index_dir=RAG_INDEX_DIR)
        print("RAG retriever loaded")
    except Exception as e:
        print(f"RAG not available: {e}")

def get_context(question):
    if retriever is None:
        return ""
    chunks = retriever.query(question, top_k=3)
    return "\n---\n".join(c["text"][:350] for c in chunks)


# ── Your own model ─────────────────────────────────────────
if "own_no_rag" in MODELS_TO_RUN or "own_rag" in MODELS_TO_RUN:
    import tiktoken
    from rag_pipeline import load_model
    own_mdl, _, own_cfg = load_model(CHECKPOINT_PATH, device)
    enc = tiktoken.get_encoding("gpt2")

    def _own_generate(prompt):
        ids = enc.encode(prompt)
        x   = torch.tensor([ids], dtype=torch.long).to(device)
        out = own_mdl.generate(x, max_new_tokens=MAX_TOKENS,
                                temperature=TEMPERATURE, top_k=50,
                                stop_token=enc.eot_token)
        return enc.decode(out[0, len(ids):].tolist()).replace("<|endoftext|>", "").strip()

    if "own_no_rag" in MODELS_TO_RUN:
        def own_no_rag(question):
            return _own_generate(f"Question: {question}\nAnswer:")
        models["own_no_rag"] = own_no_rag

    if "own_rag" in MODELS_TO_RUN:
        def own_rag(question):
            ctx = get_context(question)
            prefix = f"Context: {ctx}\n\n" if ctx else ""
            return _own_generate(f"{prefix}Question: {question}\nAnswer:")
        models["own_rag"] = own_rag

    print("Own model loaded")


# ── flan-t5-base + RAG ─────────────────────────────────────
if "flan_rag" in MODELS_TO_RUN:
    from transformers import T5ForConditionalGeneration, T5Tokenizer
    _tok = T5Tokenizer.from_pretrained("google/flan-t5-base")
    _mdl = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base").to(device)
    _mdl.eval()
    def flan_rag(question):
        ctx = get_context(question)
        if ctx:
            prompt = f"Context: {ctx}\n\nQuestion: {question}\n\nAnswer:"
        else:
            prompt = f"Question: {question}\n\nAnswer:"
        inputs = _tok(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
        with torch.no_grad():
            out_ids = _mdl.generate(**inputs, max_new_tokens=MAX_TOKENS,
                                    do_sample=True, temperature=TEMPERATURE)
        return _tok.decode(out_ids[0], skip_special_tokens=True)
    models["flan_rag"] = flan_rag
    print("flan-t5-base loaded")


# ── GPT-4 via API ──────────────────────────────────────────
if "gpt4" in MODELS_TO_RUN:
    import openai
    gpt_client = openai.OpenAI(api_key=OPENAI_API_KEY)
    def gpt4(question):
        resp = gpt_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a helpful ML teaching assistant."},
                {"role": "user",   "content": question},
            ],
            temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
        )
        return resp.choices[0].message.content
    models["gpt4"] = gpt4
    print("GPT-4 client ready")


print(f"\nLoaded {len(models)} model(s): {list(models.keys())}")


In [ ]:
# ═══════════════════════════════════════════════════════════
# COMPARE: run all questions through all models
# ═══════════════════════════════════════════════════════════

results = {}  # question → {model_name: answer}

for q in QUESTIONS:
    results[q] = {}
    for name, fn in models.items():
        print(f"Running {name} on: {q[:50]}...")
        try:
            results[q][name] = fn(q)
        except Exception as e:
            results[q][name] = f"[ERROR: {e}]"

print("\nDone.")

In [ ]:
# ═══════════════════════════════════════════════════════════
# DISPLAY: print results side by side
# ═══════════════════════════════════════════════════════════

for q, answers in results.items():
    print(f"\n{'═'*65}")
    print(f"Q: {q}")
    print('═'*65)
    for name, ans in answers.items():
        print(f"\n  [{name.upper()}]")
        # Word-wrap at ~70 chars for readability
        for line in ans.split("\n"):
            while len(line) > 70:
                print("    " + line[:70])
                line = line[70:]
            print("    " + line)
    print()

In [ ]:
# ═══════════════════════════════════════════════════════════
# QUICK SEMANTIC SIMILARITY SCORES
# Compare each model's output against GPT-4's as a soft proxy
# for answer quality (cosine similarity of sentence embeddings).
# Requires: pip install sentence-transformers
# ═══════════════════════════════════════════════════════════
try:
    from sentence_transformers import SentenceTransformer
    import numpy as np
    emb_model = SentenceTransformer("all-MiniLM-L6-v2")

    if "gpt4" not in results.get(QUESTIONS[0], {}):
        print("GPT-4 not run — using flan_rag as the reference instead.")
        ref_key = "flan_rag"
    else:
        ref_key = "gpt4"

    print(f"\nSemantic similarity vs {ref_key} (0=different, 1=identical):")
    print(f"{'Question':<40} ", end="")
    other_models = [m for m in models if m != ref_key]
    for m in other_models:
        print(f"{m:>15}", end="")
    print()
    print('─' * (40 + 16 * len(other_models)))

    for q in QUESTIONS:
        ref_ans = results[q].get(ref_key, "")
        if not ref_ans:
            continue
        print(f"{q[:39]:<40} ", end="")
        ref_vec = emb_model.encode([ref_ans], normalize_embeddings=True)[0]
        for m in other_models:
            other_ans = results[q].get(m, "")
            if other_ans:
                other_vec = emb_model.encode([other_ans], normalize_embeddings=True)[0]
                sim = float(np.dot(ref_vec, other_vec))
                print(f"{sim:>15.3f}", end="")
            else:
                print(f"{'—':>15}", end="")
        print()

except ImportError:
    print("sentence-transformers not installed — skipping similarity scores.")
    print("Run: pip install sentence-transformers")